In [1]:
import torch
import torch.nn as nn
import torch.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [50]:
X = torch.arange(0, 100, 0.1).view(-1, 1)
Y = 5*X + 4
W = nn.Linear(in_features= 1, out_features=1)
eps=1e-6
learning_rate = 0.5

x_mean = X.mean()
y_mean = Y.mean()
x_std = X.std()
y_std = Y.std()

def myloss_mse(predictions, target):
    err = predictions - target
    return (err ** 2).mean()

def forward(input):
    x_n = (input - x_mean) / (x_std + eps)
    return W(x_n)

def demorm_value(input_n, input_mean, input_std):
    input = (input_n * (input_std + eps)) + input_mean
    return input

def denorm_params(w_n):
    with torch.no_grad():
        w = w_n.weight * y_std / x_std
        b = y_mean + y_std * w_n.bias - w.squeeze(1) * x_mean

        W_phys = nn.Linear(in_features=1, out_features=1)
        W_phys.weight.copy_(w)
        W_phys.bias.copy_(b)

    return W_phys

def inference(x, ):
    prediction = forward(x)
    return prediction

params = [*W.parameters()]
opt = torch.optim.SGD(params, lr = learning_rate)

y_n = (Y - y_mean) / (y_std + eps)

for epoch in range(10):

    opt.zero_grad()

    prediction = forward(X)
    loss = myloss_mse(prediction, y_n)
    loss.backward()

    opt.step()

    W_phys = denorm_params(W)
    print(f"epoch: {epoch} loss: {loss.item():.{7}f} w: {W_phys.weight.item():.{2}f} b: {W_phys.bias.item():.{2}f}")





epoch: 0 loss: 2.4848828 w: 4.99 b: 4.33
epoch: 1 loss: 0.0000018 w: 5.00 b: 4.00
epoch: 2 loss: 0.0000000 w: 5.00 b: 4.00
epoch: 3 loss: 0.0000000 w: 5.00 b: 4.00
epoch: 4 loss: 0.0000000 w: 5.00 b: 4.00
epoch: 5 loss: 0.0000000 w: 5.00 b: 4.00
epoch: 6 loss: 0.0000000 w: 5.00 b: 4.00
epoch: 7 loss: 0.0000000 w: 5.00 b: 4.00
epoch: 8 loss: 0.0000000 w: 5.00 b: 4.00
epoch: 9 loss: 0.0000000 w: 5.00 b: 4.00


In [19]:
p = [list(W.parameters())]